In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split


SEED = 23
TARGET = "hours_spent"
OUTPUT_DIR = Path("residual_analysis")

OUTPUT_DIR.mkdir(exist_ok=True)

print("=" * 65)
print("RESIDUAL ANALYSIS")
print("=" * 65)


data = pd.read_csv("train.csv")

data = data.replace([np.inf, -np.inf], np.nan)
data = data.dropna(subset=[TARGET])
data = data[data[TARGET] >= 0].reset_index(drop=True)

y_raw = data[TARGET].astype(float)
y_log = np.log1p(y_raw)

excluded_columns = [
    TARGET,
    "log_hours",
    "hours_bucket"
]

X = data.drop(
    columns=excluded_columns,
    errors="ignore"
).copy()

print(f"Rows: {len(data)}")
print(f"Features: {X.shape[1]}")

X_train, X_val, y_train_log, y_val_log, y_train_raw, y_val_raw = (
    train_test_split(
        X,
        y_log,
        y_raw,
        test_size=0.2,
        random_state=SEED
    )
)

X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)

y_train_log = y_train_log.reset_index(drop=True)
y_val_log = y_val_log.reset_index(drop=True)

y_train_raw = y_train_raw.reset_index(drop=True)
y_val_raw = y_val_raw.reset_index(drop=True)

categorical_features = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numerical_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

for col in categorical_features:
    X_train[col] = X_train[col].fillna("missing").astype(str)
    X_val[col] = X_val[col].fillna("missing").astype(str)

for col in numerical_features:
    X_train[col] = pd.to_numeric(
        X_train[col],
        errors="coerce"
    )

    X_val[col] = pd.to_numeric(
        X_val[col],
        errors="coerce"
    )

    median = X_train[col].median()

    if pd.isna(median):
        median = 0

    X_train[col] = X_train[col].fillna(median)
    X_val[col] = X_val[col].fillna(median)

print(f"Categorical features: {len(categorical_features)}")
print(f"Numerical features: {len(numerical_features)}")

print("\nTraining general model...")

model = CatBoostRegressor(
    iterations=4000,
    depth=7,
    learning_rate=0.03,
    l2_leaf_reg=5,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=SEED,
    verbose=200,
    allow_writing_files=False
)

model.fit(
    X_train,
    y_train_log,
    cat_features=categorical_features,
    eval_set=(X_val, y_val_log),
    early_stopping_rounds=200,
    verbose=200
)

print(f"\nBest iteration: {model.best_iteration_}")


# ============================================================
# 5. GENERATE VALIDATION PREDICTIONS
# ============================================================

predicted_log = model.predict(X_val)

predicted_raw = np.maximum(
    np.expm1(predicted_log),
    0
)

actual_log = np.log1p(y_val_raw)

residual = actual_log - predicted_log

absolute_residual = np.abs(residual)

rmsle = np.sqrt(np.mean(residual ** 2))

print("\nValidation RMSLE:", round(rmsle, 6))

results = X_val.copy()

results["actual_hours"] = y_val_raw
results["predicted_hours"] = predicted_raw

results["actual_log"] = actual_log
results["predicted_log"] = predicted_log

results["residual"] = residual
results["absolute_residual"] = absolute_residual

results["actual_to_predicted_ratio"] = (
    (y_val_raw + 0.01) /
    (predicted_raw + 0.01)
)

results["percentage_error"] = (
    (predicted_raw - y_val_raw) /
    (y_val_raw + 0.01)
) * 100

results["error_type"] = np.select(
    [
        residual > 0,
        residual < 0
    ],
    [
        "Underpredicted",
        "Overpredicted"
    ],
    default="Exact"
)

results.to_csv(
    OUTPUT_DIR / "validation_predictions.csv",
    index=False
)

underpredicted = results.sort_values(
    "residual",
    ascending=False
)

overpredicted = results.sort_values(
    "residual",
    ascending=True
)

underpredicted.head(50).to_csv(
    OUTPUT_DIR / "most_underpredicted.csv",
    index=False
)

overpredicted.head(50).to_csv(
    OUTPUT_DIR / "most_overpredicted.csv",
    index=False
)

print("\nError distribution:")
print(results["error_type"].value_counts())

print("\nMost underpredicted projects:")
print(
    results.nlargest(
        10,
        "residual"
    )[
        [
            "actual_hours",
            "predicted_hours",
            "residual",
            "actual_to_predicted_ratio"
        ]
    ].to_string(index=False)
)

print("\nMost overpredicted projects:")
print(
    results.nsmallest(
        10,
        "residual"
    )[
        [
            "actual_hours",
            "predicted_hours",
            "residual",
            "actual_to_predicted_ratio"
        ]
    ].to_string(index=False)
)

under_threshold = results["residual"].quantile(0.80)
over_threshold = results["residual"].quantile(0.20)

results["residual_group"] = np.select(
    [
        results["residual"] >= under_threshold,
        results["residual"] <= over_threshold
    ],
    [
        "Strong underprediction",
        "Strong overprediction"
    ],
    default="Middle"
)

group_summary = results.groupby(
    "residual_group"
).agg(
    count=("residual", "size"),
    mean_actual=("actual_hours", "mean"),
    median_actual=("actual_hours", "median"),
    mean_predicted=("predicted_hours", "mean"),
    median_predicted=("predicted_hours", "median"),
    mean_residual=("residual", "mean"),
    mean_absolute_residual=("absolute_residual", "mean")
).reset_index()

group_summary.to_csv(
    OUTPUT_DIR / "residual_group_summary.csv",
    index=False
)

print("\nResidual group summary:")
print(group_summary.to_string(index=False))

numeric_pattern_rows = []

under_group = results[
    results["residual_group"] == "Strong underprediction"
]

over_group = results[
    results["residual_group"] == "Strong overprediction"
]

for col in numerical_features:

    overall = X_val[col]

    under_values = under_group[col]
    over_values = over_group[col]

    overall_mean = overall.mean()
    overall_std = overall.std()

    if pd.isna(overall_std) or overall_std == 0:
        overall_std = 1

    numeric_pattern_rows.append({
        "feature": col,

        "overall_mean": overall_mean,
        "overall_median": overall.median(),

        "under_mean": under_values.mean(),
        "under_median": under_values.median(),

        "over_mean": over_values.mean(),
        "over_median": over_values.median(),

        "under_mean_difference": (
            under_values.mean() - overall_mean
        ),

        "over_mean_difference": (
            over_values.mean() - overall_mean
        ),

        "under_standardized_difference": (
            under_values.mean() - overall_mean
        ) / overall_std,

        "over_standardized_difference": (
            over_values.mean() - overall_mean
        ) / overall_std,

        "under_missing_pct": under_values.isna().mean() * 100,
        "over_missing_pct": over_values.isna().mean() * 100
    })

numeric_patterns = pd.DataFrame(numeric_pattern_rows)

if not numeric_patterns.empty:
    numeric_patterns["max_absolute_difference"] = numeric_patterns[
        [
            "under_standardized_difference",
            "over_standardized_difference"
        ]
    ].abs().max(axis=1)

    numeric_patterns = numeric_patterns.sort_values(
        "max_absolute_difference",
        ascending=False
    )

numeric_patterns.to_csv(
    OUTPUT_DIR / "numerical_feature_patterns.csv",
    index=False
)

print("\nNumerical features with strongest residual-group differences:")

if not numeric_patterns.empty:
    print(
        numeric_patterns[
            [
                "feature",
                "under_standardized_difference",
                "over_standardized_difference"
            ]
        ].head(20).to_string(index=False)
    )

categorical_pattern_rows = []

for col in categorical_features:

    overall_counts = X_val[col].value_counts(normalize=True)

    under_counts = under_group[col].value_counts(normalize=True)
    over_counts = over_group[col].value_counts(normalize=True)

    categories = set(overall_counts.index)
    categories.update(under_counts.index)
    categories.update(over_counts.index)

    for category in categories:

        overall_pct = overall_counts.get(category, 0)
        under_pct = under_counts.get(category, 0)
        over_pct = over_counts.get(category, 0)

        categorical_pattern_rows.append({
            "feature": col,
            "category": category,

            "overall_pct": overall_pct * 100,
            "under_pct": under_pct * 100,
            "over_pct": over_pct * 100,

            "under_lift": (
                under_pct / overall_pct
                if overall_pct > 0 else np.nan
            ),

            "over_lift": (
                over_pct / overall_pct
                if overall_pct > 0 else np.nan
            ),

            "under_count": int(
                (under_group[col] == category).sum()
            ),

            "over_count": int(
                (over_group[col] == category).sum()
            )
        })

categorical_patterns = pd.DataFrame(
    categorical_pattern_rows
)

if not categorical_patterns.empty:

    categorical_patterns["max_lift_difference"] = (
        categorical_patterns[
            ["under_lift", "over_lift"]
        ].max(axis=1)
    )

    categorical_patterns = categorical_patterns.sort_values(
        "max_lift_difference",
        ascending=False
    )

categorical_patterns.to_csv(
    OUTPUT_DIR / "categorical_feature_patterns.csv",
    index=False
)

print("\nCategorical patterns with strongest lifts:")

if not categorical_patterns.empty:
    print(
        categorical_patterns[
            [
                "feature",
                "category",
                "overall_pct",
                "under_pct",
                "over_pct",
                "under_lift",
                "over_lift"
            ]
        ].head(25).to_string(index=False)
    )

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.get_feature_importance()
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

importance.to_csv(
    OUTPUT_DIR / "feature_importance.csv",
    index=False
)

print("\nTop 25 CatBoost features:")
print(importance.head(25).to_string(index=False))


# ============================================================
# 12. FEATURE IMPORTANCE VS RESIDUAL PATTERNS
# ============================================================

combined_patterns = importance.merge(
    numeric_patterns,
    on="feature",
    how="left"
)

combined_patterns = combined_patterns.merge(
    categorical_patterns.groupby("feature")[
        "max_lift_difference"
    ].max().rename("categorical_max_lift"),
    on="feature",
    how="left"
)

combined_patterns.to_csv(
    OUTPUT_DIR / "combined_feature_analysis.csv",
    index=False
)

bins = [
    0,
    1,
    3,
    8,
    20,
    50,
    100,
    140,
    np.inf
]

labels = [
    "0-1",
    "1-3",
    "3-8",
    "8-20",
    "20-50",
    "50-100",
    "100-140",
    "140+"
]

results["actual_range"] = pd.cut(
    results["actual_hours"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

range_rows = []

for label, group in results.groupby(
    "actual_range",
    observed=False
):

    if len(group) == 0:
        continue

    group_rmsle = np.sqrt(
        np.mean(group["residual"] ** 2)
    )

    range_rows.append({
        "actual_range": str(label),
        "count": len(group),
        "rmsle": group_rmsle,
        "actual_mean": group["actual_hours"].mean(),
        "predicted_mean": group["predicted_hours"].mean(),
        "mean_residual": group["residual"].mean()
    })

range_summary = pd.DataFrame(range_rows)

range_summary.to_csv(
    OUTPUT_DIR / "target_range_errors.csv",
    index=False
)

print("\nRMSLE by actual target range:")
print(range_summary.to_string(index=False))

plt.figure(figsize=(9, 7))

plt.scatter(
    results["actual_hours"],
    results["predicted_hours"],
    alpha=0.5
)

max_value = max(
    results["actual_hours"].max(),
    results["predicted_hours"].max()
)

plt.plot(
    [0, max_value],
    [0, max_value],
    linestyle="--"
)

plt.xlabel("Actual hours")
plt.ylabel("Predicted hours")
plt.title("Actual vs Predicted Hours")

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "actual_vs_predicted.png",
    dpi=200
)
plt.close()


# Log residual vs predicted log target

plt.figure(figsize=(9, 7))

plt.scatter(
    predicted_log,
    residual,
    alpha=0.5
)

plt.axhline(0, linestyle="--")

plt.xlabel("Predicted log hours")
plt.ylabel("Log residual")
plt.title("Residuals vs Predicted Log Hours")

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "residuals_vs_predictions.png",
    dpi=200
)
plt.close()


# Residual distribution

plt.figure(figsize=(9, 6))

plt.hist(
    residual,
    bins=40
)

plt.axvline(0, linestyle="--")

plt.xlabel("Log residual")
plt.ylabel("Frequency")
plt.title("Residual Distribution")

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "residual_distribution.png",
    dpi=200
)
plt.close()


# Feature importance

top_features = importance.head(20).sort_values(
    "importance",
    ascending=True
)

plt.figure(figsize=(10, 8))

plt.barh(
    top_features["feature"],
    top_features["importance"]
)

plt.xlabel("Importance")
plt.title("Top 20 CatBoost Features")

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "feature_importance.png",
    dpi=200
)
plt.close()


# Actual target range RMSLE

plt.figure(figsize=(9, 6))

plt.bar(
    range_summary["actual_range"],
    range_summary["rmsle"]
)

plt.xlabel("Actual hours range")
plt.ylabel("RMSLE")
plt.title("RMSLE by Actual Target Range")

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "rmsle_by_target_range.png",
    dpi=200
)
plt.close()
summary = pd.DataFrame([{
    "validation_rmsle": rmsle,
    "validation_rows": len(results),
    "underpredicted_count": int(
        (results["residual"] > 0).sum()
    ),
    "overpredicted_count": int(
        (results["residual"] < 0).sum()
    ),
    "mean_residual": residual.mean(),
    "mean_absolute_residual": absolute_residual.mean(),
    "median_absolute_residual": absolute_residual.median(),
    "best_iteration": model.best_iteration_
}])

summary.to_csv(
    OUTPUT_DIR / "summary.csv",
    index=False
)

print("\n" + "=" * 65)
print("RESIDUAL ANALYSIS COMPLETE")
print("=" * 65)

print(f"Validation RMSLE: {rmsle:.6f}")
print(f"Results saved to: {OUTPUT_DIR.resolve()}")

print("\nGenerated files:")

for file in sorted(OUTPUT_DIR.iterdir()):
    print(file.name)
